<a href="https://colab.research.google.com/github/SakshyamAryal077/FOUNDATION-MODEL-DOWNSTREAM-PREDICTION/blob/main/Median_beats_creation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ================================================================
# ACS19955 — REPRESENTATIVE 12-LEAD MEDIAN BEAT GENERATION
# SINGLE-CELL PIPELINE
#
# Input:
#   train_fixed.h5
#   validation.h5
#
# Output:
#   train_median_beats.h5
#   validation_median_beats.h5
#   train_median_qc.csv
#   validation_median_qc.csv
#
# Expected output signal shape:
#       (N, 12, 400)
#
# Median beat window:
#       -350 ms to +450 ms around R peak
#       175 samples before + 225 samples after = 400 samples @ 500 Hz
#
# IMPORTANT:
#   These are representative median beats DERIVED from the existing
#   10-s ECG signals. They are NOT the proprietary .med files.
# ================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import h5py

from scipy.signal import find_peaks
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")


# ================================================================
# 1. PATHS
# ================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ECG_Project/ACS19955"
)
PROJECT_ROOT1 = Path(
    "/content/drive/MyDrive/ECG_Project"
)

PROCESSED_DIR = (
    PROJECT_ROOT1 /
    "results" /
    "preprocessing" /
    "processed"
)

TRAIN_H5 = PROCESSED_DIR / "train_fixed.h5"
VAL_H5   = PROCESSED_DIR / "validation.h5"

TRAIN_CSV = PROCESSED_DIR / "train_omi.csv"
VAL_CSV   = PROCESSED_DIR / "validation_omi.csv"

OUTPUT_DIR = PROCESSED_DIR / "median_beats"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_OUT = OUTPUT_DIR / "train_median_beats.h5"
VAL_OUT   = OUTPUT_DIR / "validation_median_beats.h5"

TRAIN_QC = OUTPUT_DIR / "train_median_qc.csv"
VAL_QC   = OUTPUT_DIR / "validation_median_qc.csv"


# ================================================================
# 2. CONFIGURATION
# ================================================================

FS = 500

# Representative beat:
# 350 ms before R + 450 ms after R
PRE_R_SEC  = 0.35
POST_R_SEC = 0.45

PRE_R  = int(PRE_R_SEC * FS)       # 175
POST_R = int(POST_R_SEC * FS)      # 225

BEAT_LEN = PRE_R + POST_R           # 400 samples

# Minimum physiologically reasonable spacing between R peaks
MIN_RR_SEC = 0.28
MIN_RR = int(MIN_RR_SEC * FS)

# Primary detection lead:
# I=0, II=1, III=2, aVR=3, aVL=4, aVF=5, V1=6 ... V6=11
PRIMARY_LEAD = 1

LEAD_NAMES = [
    "I", "II", "III",
    "aVR", "aVL", "aVF",
    "V1", "V2", "V3",
    "V4", "V5", "V6"
]


print("=" * 80)
print("ACS19955 MEDIAN-BEAT GENERATION")
print("=" * 80)

print("Train H5 :", TRAIN_H5)
print("Val H5   :", VAL_H5)
print("Output   :", OUTPUT_DIR)
print()
print("Sampling frequency :", FS, "Hz")
print("Pre-R samples       :", PRE_R)
print("Post-R samples      :", POST_R)
print("Median beat length  :", BEAT_LEN)
print()


# ================================================================
# 3. VERIFY INPUT FILES
# ================================================================

required = [
    TRAIN_H5,
    VAL_H5,
    TRAIN_CSV,
    VAL_CSV
]

for p in required:
    if not p.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{p}"
        )

print("All required files found.\n")


# ================================================================
# 4. UTILITY — RECORD ID CLEANING
# ================================================================

def clean_record_id(x):

    if isinstance(x, bytes):
        x = x.decode()

    x = str(x).strip()

    for ext in [
        ".dat", ".hea", ".med",
        ".csv", ".npy", ".npz"
    ]:
        if x.lower().endswith(ext):
            x = x[:-len(ext)]

    return x


# ================================================================
# 5. R-PEAK DETECTION
#
# We primarily use Lead II.
# If detection is weak, the algorithm automatically tries all leads
# and selects the lead producing the most plausible QRS sequence.
#
# Detection is based on absolute waveform amplitude so both upright
# and inverted QRS complexes can be identified.
# ================================================================

def detect_r_peaks_single_lead(signal, fs=500):

    signal = np.asarray(
        signal,
        dtype=np.float32
    )

    signal = np.nan_to_num(signal)

    if signal.std() < 1e-8:
        return np.array([], dtype=int), 0.0

    # Normalize only for peak detection
    s = (
        signal - np.median(signal)
    ) / (np.std(signal) + 1e-8)

    qrs_energy = np.abs(s)

    # Adaptive prominence
    prominence = max(
        0.45,
        np.percentile(qrs_energy, 75) * 0.35
    )

    peaks, properties = find_peaks(
        qrs_energy,
        distance=int(0.28 * fs),
        prominence=prominence
    )

    if len(peaks) == 0:
        return peaks.astype(int), 0.0

    # Remove boundary peaks that cannot provide full beat window
    peaks = peaks[
        (peaks >= PRE_R) &
        (peaks < len(signal) - POST_R)
    ]

    if len(peaks) == 0:
        return peaks.astype(int), 0.0

    # Refine each R location locally
    refined = []

    search_radius = int(0.04 * fs)  # ±40 ms

    for p in peaks:

        left = max(0, p - search_radius)
        right = min(len(signal), p + search_radius + 1)

        local = np.abs(s[left:right])

        if len(local) == 0:
            continue

        rp = left + np.argmax(local)

        if (
            rp >= PRE_R and
            rp < len(signal) - POST_R
        ):
            refined.append(rp)

    refined = np.asarray(
        sorted(set(refined)),
        dtype=int
    )

    if len(refined) == 0:
        return refined, 0.0

    # Detection score
    amplitudes = np.abs(s[refined])

    score = (
        len(refined) *
        float(np.median(amplitudes))
    )

    return refined, score


# ================================================================
# 6. MULTI-LEAD FALLBACK DETECTOR
# ================================================================

def detect_best_r_peaks(ecg):

    ecg = np.asarray(
        ecg,
        dtype=np.float32
    )

    if ecg.shape[0] != 12 and ecg.shape[1] == 12:
        ecg = ecg.T

    if ecg.shape[0] != 12:
        raise ValueError(
            f"Expected 12-lead ECG, got {ecg.shape}"
        )

    # ------------------------------------------------------------
    # Try Lead II first
    # ------------------------------------------------------------

    primary_peaks, primary_score = (
        detect_r_peaks_single_lead(
            ecg[PRIMARY_LEAD],
            FS
        )
    )

    # Normally a 10-s ECG should contain roughly 4–20 beats.
    if 3 <= len(primary_peaks) <= 25:

        return (
            primary_peaks,
            PRIMARY_LEAD,
            "lead_II"
        )

    # ------------------------------------------------------------
    # Fallback: search all 12 leads
    # ------------------------------------------------------------

    best_peaks = primary_peaks
    best_lead = PRIMARY_LEAD
    best_score = primary_score

    for lead_idx in range(12):

        peaks, score = detect_r_peaks_single_lead(
            ecg[lead_idx],
            FS
        )

        n = len(peaks)

        # Reward plausible heartbeat counts
        if 3 <= n <= 25:
            score *= 2.0

        if score > best_score:

            best_score = score
            best_peaks = peaks
            best_lead = lead_idx

    return (
        best_peaks,
        best_lead,
        "multi_lead_fallback"
    )


# ================================================================
# 7. EXTRACT ALL 12-LEAD HEARTBEATS
# ================================================================

def extract_beats(ecg, r_peaks):

    beats = []

    for r in r_peaks:

        start = r - PRE_R
        end = r + POST_R

        if (
            start >= 0 and
            end <= ecg.shape[1]
        ):

            beat = ecg[:, start:end]

            if beat.shape == (12, BEAT_LEN):

                beats.append(beat)

    if len(beats) == 0:
        return None

    return np.stack(
        beats,
        axis=0
    ).astype(np.float32)


# ================================================================
# 8. REMOVE EXTREME / BAD BEATS
#
# Median itself is robust, but this removes very distorted beats
# before computing the representative beat.
# ================================================================

def reject_outlier_beats(beats):

    if beats is None or len(beats) <= 2:
        return beats

    # Initial median template
    template = np.median(
        beats,
        axis=0
    )

    distances = []

    for beat in beats:

        d = np.mean(
            np.abs(
                beat - template
            )
        )

        distances.append(d)

    distances = np.asarray(distances)

    med = np.median(distances)

    mad = np.median(
        np.abs(distances - med)
    ) + 1e-8

    threshold = med + 3.5 * mad

    keep = distances <= threshold

    cleaned = beats[keep]

    # Never discard everything
    if len(cleaned) == 0:
        return beats

    return cleaned


# ================================================================
# 9. GENERATE REPRESENTATIVE MEDIAN BEAT
# ================================================================

def create_median_beat(ecg):

    ecg = np.asarray(
        ecg,
        dtype=np.float32
    )

    if ecg.shape[0] != 12 and ecg.shape[1] == 12:
        ecg = ecg.T

    ecg = np.nan_to_num(
        ecg,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    r_peaks, detection_lead, method = (
        detect_best_r_peaks(ecg)
    )

    beats = extract_beats(
        ecg,
        r_peaks
    )

    # ------------------------------------------------------------
    # Fallback when R detection completely fails
    #
    # We retain a central 400-sample section so dataset alignment
    # is never broken. Such records are marked in the QC file.
    # ------------------------------------------------------------

    fallback = False

    if beats is None or len(beats) == 0:

        fallback = True

        center = ecg.shape[1] // 2

        start = center - PRE_R
        end = center + POST_R

        median_beat = ecg[
            :,
            start:end
        ]

        if median_beat.shape[1] != BEAT_LEN:

            median_beat = np.zeros(
                (12, BEAT_LEN),
                dtype=np.float32
            )

        n_detected = 0
        n_used = 0

    else:

        n_detected = len(beats)

        beats_clean = reject_outlier_beats(
            beats
        )

        n_used = len(beats_clean)

        # --------------------------------------------------------
        # MEDIAN ACROSS HEARTBEATS
        # Shape:
        # beats = (Nbeats, 12, 400)
        #
        # result = (12, 400)
        # --------------------------------------------------------

        median_beat = np.median(
            beats_clean,
            axis=0
        )

    median_beat = np.asarray(
        median_beat,
        dtype=np.float32
    )

    return (
        median_beat,
        {
            "n_r_peaks": len(r_peaks),
            "n_beats_detected": n_detected,
            "n_beats_used": n_used,
            "detection_lead": LEAD_NAMES[
                detection_lead
            ],
            "detection_method": method,
            "fallback": fallback
        }
    )


# ================================================================
# 10. PROCESS ONE H5 DATASET
# ================================================================

def create_median_h5(
    input_h5,
    output_h5,
    qc_csv,
    split_name
):

    print("\n" + "=" * 80)
    print(f"PROCESSING {split_name}")
    print("=" * 80)

    qc_rows = []

    with h5py.File(
        input_h5,
        "r"
    ) as fin:

        print("Input keys:", list(fin.keys()))

        signals = fin["signals"]
        ids = fin["record_ids"]

        N = len(signals)

        print("Input signals:", signals.shape)
        print("Records:", N)

        with h5py.File(
            output_h5,
            "w"
        ) as fout:

            # ----------------------------------------------------
            # Median ECG
            # ----------------------------------------------------

            median_ds = fout.create_dataset(
                "signals",
                shape=(
                    N,
                    12,
                    BEAT_LEN
                ),
                dtype="float32",
                chunks=(
                    1,
                    12,
                    BEAT_LEN
                ),
                compression="gzip",
                compression_opts=4
            )

            # ----------------------------------------------------
            # Record IDs
            # ----------------------------------------------------

            record_ds = fout.create_dataset(
                "record_ids",
                shape=(N,),
                dtype=h5py.string_dtype(
                    encoding="utf-8"
                )
            )

            # ----------------------------------------------------
            # R-peak count
            # ----------------------------------------------------

            rpeak_ds = fout.create_dataset(
                "n_r_peaks",
                shape=(N,),
                dtype="int16"
            )

            # ----------------------------------------------------
            # Number of beats actually used
            # ----------------------------------------------------

            used_ds = fout.create_dataset(
                "n_beats_used",
                shape=(N,),
                dtype="int16"
            )

            # ----------------------------------------------------
            # QC valid/fallback flag
            # ----------------------------------------------------

            fallback_ds = fout.create_dataset(
                "fallback",
                shape=(N,),
                dtype="uint8"
            )

            for i in tqdm(
                range(N),
                desc=split_name
            ):

                ecg = signals[i]

                rid = clean_record_id(
                    ids[i]
                )

                median_beat, qc = (
                    create_median_beat(
                        ecg
                    )
                )

                median_ds[i] = median_beat

                record_ds[i] = rid

                rpeak_ds[i] = qc[
                    "n_r_peaks"
                ]

                used_ds[i] = qc[
                    "n_beats_used"
                ]

                fallback_ds[i] = int(
                    qc["fallback"]
                )

                qc_rows.append({
                    "index": i,
                    "record_id": rid,
                    **qc
                })

            # ----------------------------------------------------
            # Metadata
            # ----------------------------------------------------

            fout.attrs[
                "sampling_frequency"
            ] = FS

            fout.attrs[
                "beat_length"
            ] = BEAT_LEN

            fout.attrs[
                "pre_r_samples"
            ] = PRE_R

            fout.attrs[
                "post_r_samples"
            ] = POST_R

            fout.attrs[
                "representation"
            ] = (
                "12-lead representative "
                "median heartbeat derived "
                "from processed 10-second ECG"
            )

            fout.attrs[
                "lead_order"
            ] = json.dumps(
                LEAD_NAMES
            )

            fout.attrs[
                "source_file"
            ] = str(input_h5)

    qc_df = pd.DataFrame(
        qc_rows
    )

    qc_df.to_csv(
        qc_csv,
        index=False
    )

    print("\nCreated:")
    print(output_h5)

    print("\nQC:")
    print(
        qc_df[
            [
                "n_r_peaks",
                "n_beats_used"
            ]
        ].describe()
    )

    fallback_count = int(
        qc_df["fallback"].sum()
    )

    print(
        "\nFallback records:",
        fallback_count,
        "/",
        len(qc_df)
    )

    return qc_df


# ================================================================
# 11. CREATE TRAIN + VALIDATION MEDIAN DATASETS
# ================================================================

train_qc = create_median_h5(
    TRAIN_H5,
    TRAIN_OUT,
    TRAIN_QC,
    "TRAIN"
)

val_qc = create_median_h5(
    VAL_H5,
    VAL_OUT,
    VAL_QC,
    "VALIDATION"
)


# ================================================================
# 12. VERIFY LABEL ORDER HAS NOT CHANGED
# ================================================================

def get_h5_ids(path):

    with h5py.File(path, "r") as f:

        return [
            clean_record_id(x)
            for x in f["record_ids"][:]
        ]


train_original_ids = get_h5_ids(
    TRAIN_H5
)

train_median_ids = get_h5_ids(
    TRAIN_OUT
)

val_original_ids = get_h5_ids(
    VAL_H5
)

val_median_ids = get_h5_ids(
    VAL_OUT
)


assert (
    train_original_ids ==
    train_median_ids
), "TRAIN ID ORDER CHANGED!"

assert (
    val_original_ids ==
    val_median_ids
), "VALIDATION ID ORDER CHANGED!"


# ================================================================
# 13. VERIFY CSV LABEL ALIGNMENT
# ================================================================

train_labels = pd.read_csv(
    TRAIN_CSV,
    dtype={
        "record_id": str
    }
)

val_labels = pd.read_csv(
    VAL_CSV,
    dtype={
        "record_id": str
    }
)

train_csv_ids = [
    clean_record_id(x)
    for x in train_labels[
        "record_id"
    ]
]

val_csv_ids = [
    clean_record_id(x)
    for x in val_labels[
        "record_id"
    ]
]

assert (
    train_csv_ids ==
    train_median_ids
), "TRAIN CSV/H5 mismatch!"

assert (
    val_csv_ids ==
    val_median_ids
), "VALIDATION CSV/H5 mismatch!"


# ================================================================
# 14. FINAL SANITY CHECK
# ================================================================

def inspect_output(path):

    with h5py.File(
        path,
        "r"
    ) as f:

        print("\nFILE:")
        print(path)

        print(
            "Signals:",
            f["signals"].shape
        )

        print(
            "Record IDs:",
            f["record_ids"].shape
        )

        print(
            "Mean:",
            float(
                np.mean(
                    f["signals"][0]
                )
            )
        )

        print(
            "Std:",
            float(
                np.std(
                    f["signals"][0]
                )
            )
        )

        print(
            "NaN:",
            bool(
                np.isnan(
                    f["signals"][0]
                ).any()
            )
        )

        print(
            "Example beats used:",
            int(
                f["n_beats_used"][0]
            )
        )


inspect_output(
    TRAIN_OUT
)

inspect_output(
    VAL_OUT
)


# ================================================================
# 15. FINAL SUMMARY
# ================================================================

print("\n" + "=" * 80)
print("MEDIAN DATASET GENERATION COMPLETE")
print("=" * 80)

print("\nTRAIN:")
print(TRAIN_OUT)

print("\nVALIDATION:")
print(VAL_OUT)

print("\nTRAIN LABELS:")
print(TRAIN_CSV)

print("\nVALIDATION LABELS:")
print(VAL_CSV)

print(
    "\nMedian ECG input shape:"
    f" (12, {BEAT_LEN})"
)

print("\nIMPORTANT:")
print(
    "Do NOT resize these median beats to 5000 samples merely "
    "to imitate the original 10-second ECG."
)

print(
    "First test whether ECGFounder accepts variable temporal "
    "length (12 x 400). Its global temporal pooling may permit it."
)

print(
    "If the architecture contains a hard-coded 5000-sample "
    "dimension, modify only the input handling rather than "
    "stretching a single heartbeat into a 10-second signal."
)

print("\nEverything completed successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ACS19955 MEDIAN-BEAT GENERATION
Train H5 : /content/drive/MyDrive/ECG_Project/results/preprocessing/processed/train_fixed.h5
Val H5   : /content/drive/MyDrive/ECG_Project/results/preprocessing/processed/validation.h5
Output   : /content/drive/MyDrive/ECG_Project/results/preprocessing/processed/median_beats

Sampling frequency : 500 Hz
Pre-R samples       : 175
Post-R samples      : 225
Median beat length  : 400

All required files found.


PROCESSING TRAIN
Input keys: ['record_ids', 'signals']
Input signals: (15273, 12, 5000)
Records: 15273


TRAIN:   0%|          | 0/15273 [00:00<?, ?it/s]


Created:
/content/drive/MyDrive/ECG_Project/results/preprocessing/processed/median_beats/train_median_beats.h5

QC:
          n_r_peaks  n_beats_used
count  15273.000000  15273.000000
mean      15.552478     13.006678
std        3.934061      4.269099
min        2.000000      2.000000
25%       12.000000     10.000000
50%       15.000000     12.000000
75%       19.000000     16.000000
max       31.000000     27.000000

Fallback records: 0 / 15273

PROCESSING VALIDATION
Input keys: ['record_ids', 'signals']
Input signals: (2685, 12, 5000)
Records: 2685


VALIDATION:   0%|          | 0/2685 [00:00<?, ?it/s]


Created:
/content/drive/MyDrive/ECG_Project/results/preprocessing/processed/median_beats/validation_median_beats.h5

QC:
         n_r_peaks  n_beats_used
count  2685.000000   2685.000000
mean     15.534078     13.025698
std       3.995626      4.350053
min       3.000000      2.000000
25%      12.000000     10.000000
50%      15.000000     12.000000
75%      19.000000     16.000000
max      29.000000     28.000000

Fallback records: 0 / 2685

FILE:
/content/drive/MyDrive/ECG_Project/results/preprocessing/processed/median_beats/train_median_beats.h5
Signals: (15273, 12, 400)
Record IDs: (15273,)
Mean: 0.04421045631170273
Std: 0.9453830122947693
NaN: False
Example beats used: 10

FILE:
/content/drive/MyDrive/ECG_Project/results/preprocessing/processed/median_beats/validation_median_beats.h5
Signals: (2685, 12, 400)
Record IDs: (2685,)
Mean: 0.018917888402938843
Std: 1.0569077730178833
NaN: False
Example beats used: 10

MEDIAN DATASET GENERATION COMPLETE

TRAIN:
/content/drive/MyDrive/EC